# CoV vs Influenza vs HD Classification

In [ ]:
import glob
from pathlib import Path

import pandas as pd

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

## load results

In [ ]:
path = "./results_HD-Flu-CoV/*_HD-Flu-CoV_5fold_results.csv"
files = glob.glob(path)
len(files)

In [ ]:
dataset_map = {
    'full': 'F',
    'half': 'H',
    'quarter': 'Q'
}
def combine_stats(list_of_paths):
    # read all results files and combine
    dfs = []
    for path in list_of_paths:
        tmp = pd.read_csv(path)

        # get model name
        tmp["model"] = Path(path).stem
        
        dfs.append(tmp)
    
    df = pd.concat(dfs).reset_index(drop=True)
    
    df['dataset'] = df['model'].str.split('_').str[2].map(dataset_map)
    df['model_size'] = df['model'].str.split('_').str[1].str.replace('M', '').astype(int)

    # get means and errors
    means = df.groupby(["model_size", "dataset"]).mean(numeric_only=True)
    sem = df.groupby(["model_size", "dataset"]).sem(numeric_only=True)

    # combine in a readable table format
    combined = means.copy().astype(object)   # the cells become "mean (± sem)" strings
    for i in means.index:
        for j in [col for col in means.columns if "test" in col]:
            combined.loc[i,j] = f"{str(round(means.loc[i,j], 4))} (\xB1 {str(round(sem.loc[i,j], 4))})"

    return df, combined

In [ ]:
full_df, summary = combine_stats(files)

## summary of Dataset-F results

In [ ]:
summary.xs("F", level="dataset")

## plot confusion matrix

`02_HD_vs_CoV_vs_Flu.py` writes the per-sequence predictions of every fold to
`results_HD-Flu-CoV/predictions/`. The matrix below pools the five folds of one model.

In [ ]:
model_name = "esm_350M_full"   # the --model_name the script was run with

preds = [f"./results_HD-Flu-CoV/predictions/{model_name}_predictions_itr{i}.csv" for i in range(5)]

def itr_col(path, itr):
    df = pd.read_csv(path)
    df["itr"] = itr
    return df

all_preds = [itr_col(preds[itr], itr) for itr in range(len(preds))]
all_preds = pd.concat(all_preds, ignore_index=True)

# drop NaNs
all_preds = all_preds[~all_preds["prediction"].isna()]

In [ ]:
# the label column is 0 = healthy donor, 1 = CoV, 2 = influenza;
# reorder the classes for the plot to CoV, influenza, healthy donor
reorder = {1: 0, 2: 1, 0: 2}
all_preds["reordered_label"] = all_preds["label"].map(reorder)
all_preds["reordered_pred"] = all_preds["prediction"].map(reorder)

cm = confusion_matrix(
    all_preds["reordered_label"],
    all_preds["reordered_pred"]
)


sns.heatmap(
    cm, annot=True, fmt="d",
    cmap=sns.light_palette(
        "#176D9C",
        as_cmap=True
    ),
    vmin=0, vmax=cm.max()
)

plt.xlabel("Predicted Label", fontsize=12)
plt.xticks([0.5,1.5,2.5],
            ['CoV-specific', 'Influenza-specific', 'Healthy Donor'],
            rotation="horizontal",
            fontsize=10)
plt.ylabel("True Label", fontsize=12)
plt.yticks([0.5,1.5,2.5],
            ['CoV-specific', 'Influenza-specific', 'Healthy Donor'],
            rotation="vertical",
            fontsize=10)

plt.savefig("multiclass_conf-matrix.pdf", bbox_inches='tight', dpi=300)